# Stage 5 - NLP

Import required libraries

In [2]:
from datasets import load_dataset
import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
import joblib
import numpy as np

First lets load and view the dataset.

In [3]:
dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
print(dataset['train'][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

Now we have to clean all of this data.

In [5]:
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

train_df.head(5)

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [6]:
train_df['text_clean'] = train_df['text'].apply(lambda x: re.sub(r'<.*?>|[^\w\s]|\b\d{5,}\b|[^\x00-\x7F]+', '', x))
train_df['text_clean'] = train_df['text_clean'].apply(lambda x: re.sub(r'(.)\1{2,}', r'\1\1', x))
test_df['text_clean'] = test_df['text'].apply(lambda x: re.sub(r'<.*?>|[^\w\s]|\b\d{5,}\b|[^\x00-\x7F]+', '', x))
test_df['text_clean'] = test_df['text_clean'].apply(lambda x: re.sub(r'(.)\1{2,}', r'\1\1', x))
train_df['text_clean'] = train_df['text_clean'].str.lower()
test_df['text_clean'] = test_df['text_clean'].str.lower()

print(train_df['text_clean'][0])

i rented i am curiousyellow from my video store because of all the controversy that surrounded it when it was first released in 1967 i also heard that at first it was seized by us customs if it ever tried to enter this country therefore being a fan of films considered controversial i really had to see this for myselfthe plot is centered around a young swedish drama student named lena who wants to learn everything she can about life in particular she wants to focus her attentions to making some sort of documentary on what the average swede thought about certain political issues such as the vietnam war and race issues in the united states in between asking politicians and ordinary denizens of stockholm about their opinions on politics she has sex with her drama teacher classmates and married menwhat kills me about i am curiousyellow is that 40 years ago this was considered pornographic really the sex and nudity scenes are few and far between even then its not shot like some cheaply made 

Now we have to tokenize it.

In [7]:
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\knico\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\knico\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\knico\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
stop_words = set(stopwords.words('english'))
def tokenize_and_remove_stopwords(text):
    tokens = word_tokenize(text)
    return [word for word in tokens if word.lower() not in stop_words]

In [9]:
train_df['tokens'] = train_df['text_clean'].apply(tokenize_and_remove_stopwords)
test_df['tokens'] = test_df['text_clean'].apply(tokenize_and_remove_stopwords)

Now we have to turn the text into numbers.

In [10]:
train_df['tokens_joined'] = train_df['tokens'].apply(lambda tokens: ' '.join(tokens))
test_df['tokens_joined'] = test_df['tokens'].apply(lambda tokens: ' '.join(tokens))

vectorizer = CountVectorizer()

X_train_counts = vectorizer.fit_transform(train_df['tokens_joined'])
X_test_counts = vectorizer.transform(test_df['tokens_joined'])

vectorizer.get_feature_names_out()

array(['00', '001', '0010', ..., 'zz', 'zzz', 'zzzz'],
      shape=(141743,), dtype=object)

Now we need to train the model. And then predict

In [11]:
model = MultinomialNB()
model.fit(X_train_counts, train_df['label'])

predictions = model.predict(X_test_counts)
accuracy_score(test_df['label'], predictions)

0.82576

Now we try with different models and vectorizers.

In [12]:
vectorizer = TfidfVectorizer()

X_train_counts = vectorizer.fit_transform(train_df['tokens_joined'])
X_test_counts = vectorizer.transform(test_df['tokens_joined'])

vectorizer.get_feature_names_out()

array(['00', '001', '0010', ..., 'zz', 'zzz', 'zzzz'],
      shape=(141743,), dtype=object)

In [13]:
model = LogisticRegression()
model.fit(X_train_counts, train_df['label'])

predictions = model.predict(X_test_counts)
accuracy_score(test_df['label'], predictions)

0.88308

We see here that we achieved better accuracy with Tfidf Vectorizer and Logistic Regression. The next step is to use the overview column in the original dataset from the first 4 stages to predict the genre.

In [16]:
df = pd.read_csv('data/movies_encoded_genres.csv')
df['overview'] = df['overview'].fillna('')
mlb = joblib.load('stage_4/mlb.pkl')

df['overview_clean'] = df['overview'].apply(lambda x: re.sub(r'<.*?>|[^\w\s]|\b\d{5,}\b|[^\x00-\x7F]+', '', x))
df['overview_clean'] = df['overview_clean'].apply(lambda x: re.sub(r'(.)\1{2,}', r'\1\1', x))

X = df['overview_clean'].apply(tokenize_and_remove_stopwords).apply(lambda tokens: ' '.join(tokens))
y = df[mlb.classes_]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

genre_model = OneVsRestClassifier(LogisticRegression())

genre_model.fit(X_train_vec, y_train)
predictions = genre_model.predict(X_test_vec)
baseline_predictions = np.zeros_like(y_test)
drama_index = list(mlb.classes_).index('Drama') 
baseline_predictions[:, drama_index] = 1

print('Baseline prediction (always Drama):', f1_score(y_test, baseline_predictions, average='samples'))
print('Model predictions:', f1_score(y_test, predictions, average='samples'))


print(classification_report(y_test, predictions, target_names=list(mlb.classes_)))

Baseline prediction (always Drama): 0.27620535714285716
Model predictions: 0.2909027777777778
                 precision    recall  f1-score   support

         Action       0.80      0.14      0.24       245
      Adventure       0.80      0.02      0.04       173
      Animation       0.00      0.00      0.00        47
         Comedy       0.80      0.31      0.44       347
          Crime       1.00      0.04      0.07       155
    Documentary       0.00      0.00      0.00        20
          Drama       0.67      0.63      0.65       440
         Family       0.00      0.00      0.00        99
        Fantasy       0.00      0.00      0.00        89
        Foreign       0.00      0.00      0.00         5
        History       0.00      0.00      0.00        37
         Horror       1.00      0.01      0.02       102
          Music       0.00      0.00      0.00        38
        Mystery       0.00      0.00      0.00        68
        Romance       0.81      0.10      0.17    

c:\Users\knico\projects\CineML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\knico\projects\CineML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\knico\projects\CineML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[

This model is not very accurate. This is due to the imbalance across the 20 genres. Some genres have only just a few movies. Lets trying balancing the classes.

In [17]:
genre_model = OneVsRestClassifier(LogisticRegression(class_weight='balanced'))

genre_model.fit(X_train_vec, y_train)
predictions = genre_model.predict(X_test_vec)

print('Baseline prediction (always Drama):', f1_score(y_test, baseline_predictions, average='samples'))
print('Model predictions:', f1_score(y_test, predictions, average='samples'))


print(classification_report(y_test, predictions, target_names=list(mlb.classes_)))

Baseline prediction (always Drama): 0.27620535714285716
Model predictions: 0.5691849296536797
                 precision    recall  f1-score   support

         Action       0.63      0.68      0.65       245
      Adventure       0.50      0.54      0.52       173
      Animation       0.50      0.43      0.46        47
         Comedy       0.68      0.66      0.67       347
          Crime       0.62      0.56      0.59       155
    Documentary       0.60      0.30      0.40        20
          Drama       0.67      0.67      0.67       440
         Family       0.51      0.58      0.54        99
        Fantasy       0.49      0.40      0.44        89
        Foreign       0.00      0.00      0.00         5
        History       0.51      0.54      0.53        37
         Horror       0.61      0.59      0.60       102
          Music       0.61      0.45      0.52        38
        Mystery       0.36      0.32      0.34        68
        Romance       0.52      0.54      0.53    

c:\Users\knico\projects\CineML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\knico\projects\CineML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\knico\projects\CineML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[

The model is now much better. We see the f1 score doubled with the class balancing. This is due to the fact that genres such as TV Movie and Foreign have little or no representatives in the dataset. That is also part of the reson the f1 score now is still only .57. There are definitely limitations to this model, but as we can see it is generally more accurate than guessing just one genre.